In [ ]:
"""
CLASSIFIER COMPARISON
XGBoost vs Random Forest vs SVM vs Logistic Regression on GSE45827
====================================================================

 "Compare XGBoost with other classifiers
to demonstrate that your chosen model is justified."

DESIGN PRINCIPLE:
  All four classifiers use the IDENTICAL preprocessing pipeline (SelectKBest
  mutual_info, k=300 -> StandardScaler), refit independently inside each CV
  fold. This isolates the comparison to the CLASSIFIER ALGORITHM ITSELF --
  nothing else differs between the four models. The XGBoost configuration
  matches Supplementary Table S1 / frozen_model.pkl EXACTLY
  (n_estimators=100, max_depth=6, learning_rate=0.3, subsample=1.0,
  colsample_bytree=1.0) so this comparison describes the same model family
  used everywhere else in the manuscript.

  RF / SVM / Logistic Regression are left at close-to-default settings
  (no aggressive hyperparameter search) -- consistent with the manuscript's
  own stated philosophy for XGBoost ("no additional hyperparameter tuning...
  minimizes overfitting risk on the small discovery cohort"). Tuning only
  the alternatives while leaving XGBoost untuned would bias the comparison;
  tuning none of them keeps it fair.

PRIMARY METRIC: 5-fold stratified cross-validation (mean +/- SD).
  A single 80/20 hold-out split is included as a SECONDARY sanity check only
  (Cell 8) -- with n=130, a hold-out test fold of ~26 samples is too small
  and too noisy to be the headline comparison metric on its own.

Run each "CELL" block as a separate cell in Google Colab.
"""

# ============================================================
# — Install dependencies (Colab)
# ============================================================
# !pip install GEOparse xgboost scikit-learn pandas numpy -q

# ============================================================
# — Imports
# ============================================================
!pip install GEOparse xgboost scikit-learn pandas numpy -q

import warnings
warnings.filterwarnings("ignore")

import GEOparse
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

VALID_SUBTYPES = ["Basal", "Her2", "Luminal A", "Luminal B"]
SEED = 42
K_FEATURES = 300

# ============================================================
#  — Load GSE45827 (discovery cohort, n=130)
# ============================================================


def load_gse45827():
    gse = GEOparse.get_GEO(geo="GSE45827", destdir="./geo_cache")
    rows, labels, sample_ids = [], [], []
    for gsm_name, gsm in gse.gsms.items():
        chars = gsm.metadata.get("characteristics_ch1", [])
        subtype = None
        for c in chars:
            if c.lower().startswith("tumor subtype"):
                subtype = c.split(":", 1)[1].strip()
        if subtype not in VALID_SUBTYPES:
            continue
        expr = gsm.table.set_index("ID_REF")["VALUE"]
        rows.append(expr)
        labels.append(subtype)
        sample_ids.append(gsm_name)
    X = pd.DataFrame(rows, index=sample_ids)
    y = pd.Series(labels, index=sample_ids, name="subtype")
    return X, y


print("Loading GSE45827...")
X, y = load_gse45827()
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Samples: {X.shape[0]}, probes: {X.shape[1]}")
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ============================================================
#  — Define the four models
# Same preprocessing (SelectKBest k=300 -> StandardScaler) wraps every model
# identically inside a Pipeline; only the final classifier differs.
# ============================================================


def make_pipeline(classifier):
    return Pipeline(
        [
            ("selector", SelectKBest(score_func=mutual_info_classif, k=K_FEATURES)),
            ("scaler", StandardScaler()),
            ("clf", classifier),
        ]
    )


models = {
    "XGBoost": make_pipeline(  
       XGBClassifier(
           n_estimators=100,
           max_depth=6,
           learning_rate=0.3,
           tree_method='hist', 
           device='cuda',           
           eval_metric="mlogloss",
           objective="multi:softprob",
           random_state=SEED,
        )
        
    ),
    "RandomForest": make_pipeline(
        RandomForestClassifier(n_estimators=100, n_jobs=4, random_state=SEED)
    ),
    "SVM": make_pipeline(SVC(probability=True, random_state=SEED)),
    "LogisticRegression": make_pipeline(
        LogisticRegression(max_iter=2000, n_jobs=4, solver="saga", random_state=SEED)
    ),
}

# ============================================================
#  — 5-fold stratified cross-validation (PRIMARY comparison metric)
# Feature selection + scaling are refit inside every fold via the Pipeline —
# no leakage between folds.
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_records = []

for name, pipe in models.items():
    fold_acc, fold_f1_macro, fold_f1_weighted, fold_auc = [], [], [], []
    for train_idx, test_idx in cv.split(X.values, y_encoded):
        X_tr, X_te = X.values[train_idx], X.values[test_idx]
        y_tr, y_te = y_encoded[train_idx], y_encoded[test_idx]

        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        y_proba = pipe.predict_proba(X_te)

        fold_acc.append(accuracy_score(y_te, y_pred))
        fold_f1_macro.append(f1_score(y_te, y_pred, average="macro"))
        fold_f1_weighted.append(f1_score(y_te, y_pred, average="weighted"))
        fold_auc.append(
            roc_auc_score(y_te, y_proba, multi_class="ovr", average="macro")
        )

    cv_records.append(
        {
            "Model": name,
            "CV_Accuracy_Mean": np.mean(fold_acc),
            "CV_Accuracy_SD": np.std(fold_acc),
            "CV_MacroF1_Mean": np.mean(fold_f1_macro),
            "CV_MacroF1_SD": np.std(fold_f1_macro),
            "CV_WeightedF1_Mean": np.mean(fold_f1_weighted),
            "CV_WeightedF1_SD": np.std(fold_f1_weighted),
            "CV_ROC_AUC_OVR_Macro_Mean": np.mean(fold_auc),
            "CV_ROC_AUC_OVR_Macro_SD": np.std(fold_auc),
        }
    )
    print(f"{name}: CV accuracy = {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}")

cv_summary_df = pd.DataFrame(cv_records)
print("\n=== 5-fold CV classifier comparison (PRIMARY result) ===")
print(cv_summary_df.round(3))

# ============================================================
#  — Save primary CV comparison table
# ============================================================

import os

OUT_DIR = "/content/repo1_classifier_comparison"
os.makedirs(OUT_DIR, exist_ok=True)
cv_summary_df.to_csv(f"{OUT_DIR}/classifier_comparison_cv_summary.csv", index=False)
print(f"Saved: {OUT_DIR}/classifier_comparison_cv_summary.csv")

# ============================================================
#  — SECONDARY sanity check: single 80/20 hold-out split
# Not the headline metric (26-sample test fold is noisy) -- included only
# as a supplementary cross-check, with a leakage audit.
# ============================================================

X_train, X_test, y_train, y_test, train_ids, test_ids = train_test_split(
    X.values, y_encoded, X.index, test_size=0.20, stratify=y_encoded, random_state=SEED
)

holdout_records = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)
    holdout_records.append(
        {
            "Model": name,
            "Holdout_Accuracy": accuracy_score(y_test, y_pred),
            "Holdout_MacroF1": f1_score(y_test, y_pred, average="macro"),
            "Holdout_WeightedF1": f1_score(y_test, y_pred, average="weighted"),
            "Holdout_ROC_AUC_OVR_Macro": roc_auc_score(
                y_test, y_proba, multi_class="ovr", average="macro"
            ),
        }
    )

holdout_df = pd.DataFrame(holdout_records)
print("\n=== Secondary hold-out sanity check (n_test=%d) ===" % len(test_ids))
print(holdout_df.round(3))

# Leakage audit — same checks as before, run every time this cell runs
train_ids_set, test_ids_set = set(train_ids), set(test_ids)
overlap = train_ids_set & test_ids_set
dup_rows_test_in_train = pd.DataFrame(X_test).apply(tuple, axis=1).isin(
    pd.DataFrame(X_train).apply(tuple, axis=1)
).sum()
print(f"\nTrain/test sample-ID overlap: {len(overlap)} (should be 0)")
print(f"Test rows exactly matching a train row: {dup_rows_test_in_train} (should be 0)")

holdout_df.to_csv(f"{OUT_DIR}/classifier_comparison_holdout_secondary.csv", index=False)
print(f"Saved: {OUT_DIR}/classifier_comparison_holdout_secondary.csv")

# ============================================================
#  — Interpretation note
# ============================================================

best_cv = cv_summary_df.loc[cv_summary_df["CV_Accuracy_Mean"].idxmax(), "Model"]
xgb_row = cv_summary_df[cv_summary_df["Model"] == "XGBoost"].iloc[0]

print(
    f"\nHighest mean CV accuracy: {best_cv}. "
    f"XGBoost CV accuracy: {xgb_row['CV_Accuracy_Mean']:.3f} +/- {xgb_row['CV_Accuracy_SD']:.3f}."
)
print(
    "If XGBoost is not the top performer by raw accuracy, that is fine to report "
    "directly -- the manuscript's justification for XGBoost rests on SHAP's "
    "axiomatic per-feature attribution (local accuracy, missingness, consistency), "
    "not on claiming best-in-class accuracy on n=130. State that explicitly rather "
    "than let a reviewer draw a less charitable conclusion."
)


09-Jul-2026 06:18:12 DEBUG utils - Directory ./geo_cache already exists. Skipping.
09-Jul-2026 06:18:12 INFO GEOparse - File already exist: using local version.
09-Jul-2026 06:18:12 INFO GEOparse - Parsing ./geo_cache/GSE45827_family.soft.gz: 
09-Jul-2026 06:18:12 DEBUG GEOparse - DATABASE: GeoMiame
09-Jul-2026 06:18:12 DEBUG GEOparse - SERIES: GSE45827
09-Jul-2026 06:18:12 DEBUG GEOparse - PLATFORM: GPL570


Loading GSE45827...


09-Jul-2026 06:18:13 DEBUG GEOparse - SAMPLE: GSM1116084
09-Jul-2026 06:18:13 DEBUG GEOparse - SAMPLE: GSM1116085
09-Jul-2026 06:18:13 DEBUG GEOparse - SAMPLE: GSM1116086
09-Jul-2026 06:18:13 DEBUG GEOparse - SAMPLE: GSM1116087
09-Jul-2026 06:18:13 DEBUG GEOparse - SAMPLE: GSM1116088
09-Jul-2026 06:18:13 DEBUG GEOparse - SAMPLE: GSM1116089
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116090
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116091
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116092
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116093
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116094
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116095
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116096
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116097
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116098
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116099
09-Jul-2026 06:18:14 DEBUG GEOparse - SAMPLE: GSM1116100
09-Jul-2026 06:18:14 DEBUG GEOp

Samples: 130, probes: 29873
Label mapping: {'Basal': np.int64(0), 'Her2': np.int64(1), 'Luminal A': np.int64(2), 'Luminal B': np.int64(3)}
XGBoost: CV accuracy = 0.923 +/- 0.042
RandomForest: CV accuracy = 0.962 +/- 0.024
SVM: CV accuracy = 0.962 +/- 0.024
LogisticRegression: CV accuracy = 0.954 +/- 0.045

=== 5-fold CV classifier comparison (PRIMARY result) ===
                Model  CV_Accuracy_Mean  CV_Accuracy_SD  CV_MacroF1_Mean  \
0             XGBoost             0.923           0.042            0.915   
1        RandomForest             0.962           0.024            0.959   
2                 SVM             0.962           0.024            0.959   
3  LogisticRegression             0.954           0.045            0.951   

   CV_MacroF1_SD  CV_WeightedF1_Mean  CV_WeightedF1_SD  \
0          0.049               0.922             0.042   
1          0.027               0.961             0.025   
2          0.029               0.961             0.026   
3          0.048      